# Session 3 : Polars - une alternative performante à Pandas

#### Master TIDE 2026-2027

1. **Données des prénoms US et FR**
2. **Données SWI de Météo France**

&copy; Francis Wolinski 2026

In [ ]:
# imports

import glob

import numpy as np
import pandas as pd

import polars as pl

*Nota bene* :
- Les **datasets** des prénoms utilisés dans ce notebook sont à télécharger vous-même
- Dans ce notebook, le code **pandas** et **polars** est écrit en *method chaining*, voir cet article pour **pandas** : https://www.stat4decision.com/fr/method-chaining-avec-la-librairie-pandas/

## 1. Données des prénoms US et FR

### 1.1 Chargement des données US

On utilise les données accessibles sur le site de la Sécurité Sociale US : https://www.ssa.gov/oact/babynames/limits.html

Télécharger le fichier zip https://www.ssa.gov/oact/babynames/names.zip et le décompresser dans un répertoire "names" accessible directement depuis le notebook.

Prénom, genre et nombre de naissances des enfants nés aux USA depuis 1880.

Le site indique :

> For each year of birth YYYY after 1879, we created a comma-delimited file called yobYYYY.txt. Each record in the individual annual files has the format "name,sex,number", where name is 2 to 15 characters, sex is M (male) or F (female) and "number" is the number of occurrences of the name. Each file is sorted first on sex and then on number of occurrences in descending order. When there is a tie on the number of occurrences, names are listed in alphabetical order. (...)
> To safeguard privacy, we restrict our list of names to those with at least 5 occurrences.

On utilise le nom des fichiers qui sont sous la forme `yobAAAA.txt` pour créer une colonne supplémentaire avec l'année.

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        .assign(year=year)
        ) for year in range(1880, 2026)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .loc[:, ["year", "name", "gender", "births"]]
      )

pdf

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns=["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32],
                include_file_paths="filename")
      # syntaxe avec keyword arguments
      .with_columns(year=pl.col("filename").str.slice(9, 4).cast(pl.UInt16))
      # syntaxe avec .alias()
      #.with_columns(pl.col("filename").str.slice(9, 4).cast(pl.UInt16).alias("year"))
      .select(["year", "name", "gender", "births"])
     )

df = lf.collect()
df

In [ ]:
# test : nécessite pyarrow
assert df.equals(pl.from_pandas(pdf))
print("OK")

In [ ]:
# schéma
lf.collect_schema()

In [ ]:
# naive query plan
lf

In [ ]:
# optimized query plan
lf.show_graph()

In [ ]:
# query plan
print(lf.explain())

#### empreinte mémoire : pandas vs polars

In [ ]:
# empreinte mémoire : pandas vs polars
print(f"pandas DataFrame : {pdf.memory_usage(deep=True).sum() / 2**20:.1f} Mio")
print(f"polars DataFrame : {df.estimated_size() / 2**20:.1f} Mio")

In [ ]:
# empreinte mémoire : index pandas
print(f"Index pandas avec drop(0) : {pdf.drop(0).index.memory_usage()}")
print(f"Index pandas avec drop(1) : {pdf.drop(1).index.memory_usage()}")

Expliquer l'écart dans **pandas** entre l'empreinte mémoire avec `.drop(0)` et celle avec `.drop(1)`.

### 1.2 Sélection des données

**Exercice n° 1**

Ajouter 2 conditions sur des colonnes :
- Le nombre de naissances doit être supérieur ou égal à 1000.
- L'année doit être supérieure ou égale à 2000.
- Comparer les 2 graphes : *naive query plan* et *optimized query plan*.

Aide :
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.filter.html
- https://docs.pola.rs/api/python/stable/reference/expressions/col.html

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        .assign(year=year)
        ) for year in range(1880, 2025)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .loc[lambda df_: (df_.births >= 1000) & (df_.year >= 2000), ["year", "name", "gender", "births"]]
       #.reset_index(drop=True)
      )

pdf

In [ ]:
pdf.info()

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns= ["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32],
                include_file_paths="filename")
      .with_columns(year=pl.col("filename").str.slice(9, 4).cast(pl.UInt16))
      # VOTRE CODE ICI
     )

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

In [ ]:
# naive query plan
lf

In [ ]:
# optimized query plan
lf.show_graph()

### 1.3 Tableaux croisés

**Exercice n° 2**

Calculer 2 tableaux croisés :
- Le nombre total de naissances par année et par genre.
- La diversité des prénoms par année et par genre.

Aide :
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.pivot.html
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.sort.html

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        .assign(year=year)
        ) for year in range(1880, 2025)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .pivot_table(index="year",
                    columns="gender",
                    values="births",
                    aggfunc="sum")
       .reset_index()
      )

pdf

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns= ["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32],
                include_file_paths="filename")
      .with_columns(year=pl.col("filename").str.slice(9, 4).cast(pl.UInt16))
      # VOTRE CODE ICI
     )

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        .assign(year=year)
        ) for year in range(1880, 2025)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .pivot_table(index="year",
                    columns="gender",
                    values="name",
                    aggfunc="count")
       .reset_index()
      )

pdf

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns= ["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32],
                include_file_paths="filename")
      .with_columns(year=pl.col("filename").str.slice(9, 4).cast(pl.UInt16))
      # VOTRE CODE ICI
     )

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

### 1.4 User-defined function et calculs d'agrégats

On définit et applique une fonction qui raccourcit les prénoms.

On calcule différents agrégats.

In [ ]:
# fonction short_name

def short_name(name):
    """
    Shorten a name, e.g. "Jo" => "Jo", "Alicia" => "A4a"

    name -- input name

    Returns a shorten name.
    """
    return name if len(name) == 2 else f"{name[0]}{len(name)-2}{name[-1]}"

short_name("Internationalisation")

**Exercice n° 3**

 - Calculer le top 10 des prénoms raccourcis sur toutes les données.
 - Calculer le top 10 des prénoms raccourcis en unicisant les prénoms auparavant.
 - Calculer le top 10 des prénoms raccourcis en tenant compte du nombre de naissances.

*Nota bene* : la méthode `.value_counts()` existe dans **polars** mais uniquement sur les DataFrames. On passe donc par un `group_by()` / `len()`.

Aide :
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.with_columns.html
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.map_elements.html
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.lazyframe.group_by.LazyGroupBy.len.html
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.top_k.html
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.unique.html
- https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.lazyframe.group_by.LazyGroupBy.agg.html

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        ) for year in range(1880, 2025)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .assign(short_name=lambda df_ : df_.name.apply(short_name))["short_name"]
       .value_counts()
       .head(10)
       .rename("len")
       .reset_index()
      )

pdf

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns= ["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32])
      # VOTRE CODE ICI
     )

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        ) for year in range(1880, 2025)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .drop_duplicates(subset=["name"])
       .assign(short_name=lambda df_ : df_.name.apply(short_name))["short_name"]
       .value_counts()
       .head(10)
       .rename("len")
       .reset_index()
      )

pdf

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns= ["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32])
      # VOTRE CODE ICI
     )

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

#### avec pandas

In [ ]:
%%time

dfs = [(pd
        .read_csv(f"names/yob{year}.txt",
                  header=None,
                  names=["name", "gender", "births"])
        ) for year in range(1880, 2025)]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .assign(short_name=lambda df_ : df_.name.apply(short_name))
       .groupby("short_name", as_index=False)["births"]
       .sum()
       .nlargest(10, "births")
       .reset_index(drop=True)
      )

pdf

#### avec polars

In [ ]:
%%time

lf = (pl
      .scan_csv("names/*.txt",
                has_header=False,
                new_columns= ["name", "gender", "births"],
                schema_overrides=[pl.String, pl.Enum(["F", "M"]), pl.UInt32])
      # VOTRE CODE ICI
     )

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

### 1.5 Chargement et transformation des prénoms FR

Fichier des prénoms : https://www.insee.fr/fr/statistiques/8595130?sommaire=8595113

Télécharger le fichier zip https://www.insee.fr/fr/statistiques/fichier/8595130/prenoms-2025-nat_csv.zip et le décompresser dans un fichier CSV qui sera placé dans le répertoire `data/`.

Genre, prénom, année et nombre de naissances des enfants nés en France depuis 1900.

Le site indique :

> Le fichier des prénoms contient des données sur les prénoms attribués aux enfants nés en France entre 1900 et 2024 au niveau national, régional et départemental. Une liste des prénoms par sexe attribués au moins 3 fois depuis 1900 est également proposée.

**Exercice n° 4**

- Charger les données des prénoms français.
- Opérer les transformations pour obtenir un LazyFrame analogue aux données US :
    - Prénoms avec des majuscules et minuscules, normalisés avec "NFD",
    - Genre "F" ou "M" de type Enum,
    - Tri identique,
    - Ordre des colonnes identique.

Aide :
- https://docs.pola.rs/api/python/stable/reference/api/polars.scan_csv.html
- https://docs.pola.rs/api/python/dev/reference/expressions/string.html
- https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.replace.html
- https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.cast.html
- https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.select.html

In [ ]:
%%time

pdf = (pd
       .read_csv("data/prenoms-2025-nat.csv",
                 sep=";",
                 header=0,
                 usecols=range(4),
                 na_values="",
                 keep_default_na=False,
                 names=["gender", "name", "year", "births"]
                )
       .assign(name=lambda df_: df_.name.str.title().str.normalize("NFD"),
               gender=lambda df_: df_.gender.map({1: "M", 2: "F"}))
       .sort_values(["year", "gender", "births", "name"], ascending=[True, True, False, True])
       .loc[:, ["year", "name", "gender", "births"]]
       .reset_index(drop=True)
      )

pdf

In [ ]:
%%time

# VOTRE CODE ICI

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))

Ensuite, il est possible de refaire les mêmes exercices qu'avec les données US.

## 2. Données SWI de Météo France

On utilise des données publiques fournies par Météo France : https://donneespubliques.meteofrance.fr/?fond=produit&id_produit=301&id_rubrique=40

Le fichier comporte 5 colonnes numériques : numéro de maille, coordonnées géographiques x et y (Lambert 93), date (au format "YYYYMM"), valeur du SWI.

Le site indique :

> Le SWI (de l’anglais *Soil Wetness Index*) est un indice d’humidité des sols documenté dans la littérature scientifique. Il représente, sur une profondeur d’environ deux mètres, l’état de la réserve en eau du sol par rapport à la réserve utile (eau disponible pour l’alimentation des plantes). Cf. atelier datacraft de mai 2024.

### 2.1 Chargement des données

**Exercice n° 5**

- Charger tous les fichiers dans un LazyFrame.
- Passer la colonne "DATE" en objet date.

Aide :
- https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.str.strptime.html

#### avec pandas

In [ ]:
%%time

dfs = [pd.read_csv(filename,
                   sep=";",
                   dtype={"DATE": str},
                   decimal=",")
       for filename in sorted(glob.glob("SWI/*.csv"))]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .assign(DATE=lambda df_: pd.to_datetime(df_.DATE, format="%Y%m"))
      )

pdf

#### avec polars

In [ ]:
%%time

# VOTRE CODE ICI

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

### 2.2 User-defined function

L'objectif est de convertir les coordonnées Lambert 93 en coordonnées WGS 84 avec les longitudes et latitudes en utilisant la librairie **lambert**.

https://github.com/willena/lambert-python

In [ ]:
# convertion des Lambert 93 (RGF93, Réseau Géodésique Français) en WGS 84 (World Geodetic System 1984)

from lambert import Lambert93, convertToWGS84Deg

lambx, lamby = 349336, 6597039
print(f"LAMBX: {lambx}, LAMBY: {lamby}")
print()

point = convertToWGS84Deg(lambx, lamby, Lambert93)
print(f"LON: {point.x}, LAT: {point.y}")

On dispose des 2 colonnes "LAMBX" et "LAMBY" et on les convertit en 2 autres colonnes "LON" et "LAT".

Dans un premier temps, on applique la conversion sur un jeu de données limité dans le temps.

**Exercice n° 6**

- Installer la librairie **lambert**
- Créer une fonction (UDF) qui transforme un dictionnaire avec les clés "LAMBX" et "LAMBY" en un autre dictionnaire avec les clés "LON" et "LAT" en utilisant la fonction `convertToWGS84Deg` de la librairie **lambert**.
- Charger les données en sélectionnant uniquement les dates postérieures à 2020.
- Ajouter au LazyFrame une nouvelle colonne de type Struct avec les champs "LON" et "LAT" en appliquant la fonction créée sur une Struct définie à la volée avec les colonnes "LAMBX" et "LAMBY".
- Remonter en colonnes du LazyFrame les champs "LON" et "LAT" de la nouvelle colonne obtenue.
- Supprimer les colonnes devenues inutiles.

Aide :
- https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.struct.html
- https://docs.pola.rs/api/python/dev/reference/api/polars.datatypes.Struct.html
- https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.unnest.html

#### avec pandas

In [ ]:
%%time

def pd_lambert93ToWGS84(lambx, lamby):
    """
    UDF which converts Lambert 93 to WSG 84.
    
    Arguments:
    lambx -- the Lambert 93 X coordinate
    lamby -- the Lambert 93 Y coordinate

    Returns a tuple with longitude and latitude.
    """
    point = convertToWGS84Deg(lambx, lamby, Lambert93)
    return point.x, point.y


dfs = [pd.read_csv(filename,
                   sep=";",
                   dtype={"DATE": str},
                   decimal=",")
       for filename in sorted(glob.glob("SWI/*.csv"))]

pdf = (pd
      .concat(dfs, ignore_index=True)
      .assign(DATE=lambda df_: pd.to_datetime(df_.DATE, format="%Y%m"))
      .loc[lambda df_: df_.DATE.dt.year >= 2020]
      .pipe(lambda df_: df_.assign(**pd.DataFrame(map(pd_lambert93ToWGS84, df_.LAMBX, df_.LAMBY), index=df_.index, columns=["LON", "LAT"])))
      .drop(["LAMBX", "LAMBY"], axis=1)
      # simple and slower version
      # .assign(point=lambda df_: df_.LAMBX.combine(df_.LAMBY, lambda x, y: convertToWGS84Deg(x, y, Lambert93)))
      # .assign(LON=lambda df_: df_.point.apply(lambda p: p.x),
      #         LAT=lambda df_: df_.point.apply(lambda p: p.y))
      # .drop(["LAMBX", "LAMBY", "point"], axis=1)
     )

pdf

#### avec polars

In [ ]:
%%time

def pl_lambert93ToWGS84(struct):
    """
    UDF which converts Lambert 93 to WSG 84.
    
    Arguments:
    struct -- a dict with keys "LAMBX" and "LAMBY"

    Returns a dict with keys "LON" and "LAT".
    """
    # VOTRE CODE ICI

# VOTRE CODE ICI

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

### 2.3 Optimisation avec une jointure

En fait, dans les données SWI, il n'y a environ que 9.000 points de longitude et de latitude différents.

Il est donc possible d'optimiser les calculs en précalculant la longitude et la latitude à partir des paires LAMBX et LAMBY <u>unicisées</u>.

Ensuite, il reste à effectuer une jointure de l'ensemble des données avec le tableau précalculé.

**Exercice n° 7**

- Charger les données non limitées dans le temps.
- Précalculer les longitudes et les latitudes à partir des "LAMBX" et "LAMBY" unicisés comme précédemment.
- Opérer une jointure sur ces 2 colonnes entre l'ensemble des données et le tableau précalculé.
- Supprimer les colonnes devenues inutiles.
- Comparer les 2 graphes : *naive query plan* et *optimized query plan*.

Aide :
- https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.join.html

In [ ]:
%%time

dfs = [pd.read_csv(filename,
                   sep=";",
                   dtype={"DATE": str},
                   decimal=",")
       for filename in sorted(glob.glob("SWI/*.csv"))]

pdf = (pd
       .concat(dfs, ignore_index=True)
       .assign(DATE=lambda df_: pd.to_datetime(df_.DATE, format="%Y%m"))
      )

grid = (pdf
        .drop(["NUMERO", "DATE", "SWI_UNIF_MENS"], axis=1)
        .drop_duplicates(subset=["LAMBX", "LAMBY"])
        .reset_index(drop=True)
        .pipe(lambda df_: df_.assign(**pd.DataFrame(map(pd_lambert93ToWGS84, df_.LAMBX, df_.LAMBY), index=df_.index, columns=["LON", "LAT"])))
       )

print(grid.shape)
print()

pdf = (pd
       .merge(pdf, grid, left_on=["LAMBX", "LAMBY"], right_on=["LAMBX", "LAMBY"], how="left")
       .drop(["LAMBX", "LAMBY"], axis=1)
      )

pdf

In [ ]:
%%time

# VOTRE CODE ICI

df = lf.collect()
df

In [ ]:
# test
assert df.equals(pl.from_pandas(pdf))
print("OK")

In [ ]:
# naive query plan
lf

In [ ]:
# optimized query plan
lf.show_graph()